# 显示包的地址


import numpy
print("ipykernel is installed!")
print(numpy.__file__)

# KNN算法

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from matplotlib.colors import ListedColormap
#导入iris数据
from sklearn.datasets import load_iris
iris = load_iris()
X=iris.data[:,:2] #只取前两列
y=iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y,random_state=42) #划分数据，random_state固定划分方式
#导入模型
from sklearn.neighbors import KNeighborsClassifier 
#训练模型
n_neighbors = 5
knn = KNeighborsClassifier(n_neighbors=n_neighbors)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)
#查看各项得分
print("y_pred",y_pred)
print("y_test",y_test)
print("score on train set", knn.score(X_train, y_train))
print("score on test set", knn.score(X_test, y_test))
print("accuracy score", accuracy_score(y_test, y_pred))

# 可视化

# 自定义colormap
def colormap():
    return mpl.colors.LinearSegmentedColormap.from_list('cmap', ['#FFC0CB','#00BFFF', '#1E90FF'], 256)

x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
axes=[x_min, x_max, y_min, y_max]
xp=np.linspace(axes[0], axes[1], 500) #均匀500的横坐标
yp=np.linspace(axes[2], axes[3],500) #均匀500个纵坐标
xx, yy=np.meshgrid(xp, yp) #生成500X500网格点
xy=np.c_[xx.ravel(), yy.ravel()] #按行拼接，规范成坐标点的格式
y_pred = knn.predict(xy).reshape(xx.shape) #训练之后平铺

# 可视化方法一
plt.figure(figsize=(15,5),dpi=100)
plt.subplot(1,2,1)
plt.contourf(xx, yy, y_pred, alpha=0.3, cmap=colormap())
#画三种类型的点
p1=plt.scatter(X[y==0,0], X[y==0, 1], color='blue',marker='^')
p2=plt.scatter(X[y==1,0], X[y==1, 1], color='green', marker='o')
p3=plt.scatter(X[y==2,0], X[y==2, 1], color='red',marker='*')
#设置注释
plt.legend([p1, p2, p3], iris['target_names'], loc='upper right',fontsize='large')
#设置标题
plt.title(f"3-Class classification (k = {n_neighbors})", fontdict={'fontsize':15} )

# 可视化方法二
plt.subplot(1,2,2)
cmap_light = ListedColormap(['pink', 'cyan', 'cornflowerblue'])
cmap_bold = ListedColormap(['darkorange', 'c', 'darkblue'])
plt.pcolormesh(xx, yy, y_pred, cmap=cmap_light)

# Plot also the training points
plt.scatter(X[:, 0], X[:, 1], c=y, cmap=cmap_bold,
                edgecolor='k', s=20)
plt.xlim(xx.min(), xx.max())
plt.ylim(yy.min(), yy.max())
plt.title(f"3-Class classification (k = {n_neighbors})" ,fontdict={'fontsize':15})
plt.show()


# KNN算法实现猫狗分类

In [17]:
import os  
from PIL import Image  
  
# 数据根目录  
data_root = 'trains\DogOrCat_trains'  # 替换为您的数据根目录路径  
  
# 遍历数据根目录下的所有文件夹  
for sub_folder in os.listdir(data_root):  
    folder_path = os.path.join(data_root, sub_folder)  
    if not os.path.isdir(folder_path):  
        continue  # 跳过非文件夹项  
      
    # 遍历文件夹下的所有图像文件  
    for image_file in os.listdir(folder_path):  
        if image_file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):  # 添加您想要处理的图像格式  
            image_path = os.path.join(folder_path, image_file)  
              
            # 打开图像  
            with Image.open(image_path) as img:  
                # 检查图像模式  
                if img.mode != 'RGB':  
                    # 转换为RGB模式  
                    rgb_img = img.convert('RGB')  
                    # 保存转换后的图像（可以覆盖原图像或保存到新位置）  
                    rgb_img.save(image_path)  # 这里会覆盖原图像，如果您不想覆盖，请指定一个新路径   
  
print("Conversion process completed.")

Conversion process completed.


In [39]:
import json  
import os  
import cv2  
import numpy as np  
from sklearn.model_selection import train_test_split  
from sklearn.neighbors import KNeighborsClassifier  
from sklearn.metrics import accuracy_score  
  
# 数据根目录  
data_root = 'trains\DogOrCat_trains'  # 替换为您的数据根目录路径  
  
# 初始化特征和标签列表  
X = []  # 存储图像特征  
y = []  # 存储标签（0表示猫，1表示狗）  
  
# 遍历数据根目录下的所有文件夹  
for sub_folder in os.listdir(data_root):  
    folder_path = os.path.join(data_root, sub_folder)  
    if not os.path.isdir(folder_path):  
        continue  # 跳过非文件夹项  
      
    # 遍历文件夹下的所有JSON文件  
    for json_file in os.listdir(folder_path):  
        if json_file.endswith('.json'):  
            json_path = os.path.join(folder_path, json_file)  
              
            # 读取JSON文件  
            with open(json_path, 'r') as f:  
                data = json.load(f)  
              
            # 提取图像路径  
            image_path = os.path.join(folder_path, data['imagePath'])  
            # print(image_path)
              
            # 读取图像  
            image = cv2.imread(image_path)  
            if image is None:  
                print(f"Warning: Unable to read image at {image_path}")  
                continue  # 跳过无法读取的图像  
              
            # 转换为灰度图像并调整大小（这里假设调整为256x256）  
            gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)  
            resized_image = cv2.resize(gray_image, (256, 256))  
              
            # 展平图像为一维特征向量  
            flat_features = resized_image.flatten()  
              
            # 将特征添加到列表中  
            X.append(flat_features)  
              
            # 根据flags确定标签  
            if data['flags']['cat']:
                # print(data['flags']['cat'])  
                label = 0  # 猫  
            elif data['flags']['dog']:  
                label = 1  # 狗  
            else:  
                # 如果flags中的cat和dog都不是true，则打印警告并跳过该图像  
                print(f"Warning: Unexpected flags in {json_path}: cat={data['flags']['cat']}, dog={data['flags']['dog']}")  
                continue  
              
            # 将标签添加到列表中  
            y.append(label)  
  
# 将特征和标签转换为NumPy数组  
X = np.array(X)  
y = np.array(y)  
  
# 拆分数据集为训练集和测试集  
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)  
  
# 训练KNN模型  
knn = KNeighborsClassifier(n_neighbors=3)  # 可以根据需要调整k值  
knn.fit(X_train, y_train)  
  
# 使用测试集进行预测  
y_pred = knn.predict(X_test)  
  
# 计算并打印准确率  
accuracy = accuracy_score(y_test, y_pred)  
print(f'Accuracy: {accuracy * 100:.2f}%')

True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
